# Featurizer Tutorial: Deep Nesting

This notebook demonstrates feature generation across multiple levels of entity relationships using a retail supply chain scenario:
- **Stores** (target, depth 0)
- **Orders** (depth 1)
- **OrderItems** (depth 2) → **Products** (depth 2)
- **Suppliers** (depth 3)

We'll see how features propagate up through the entity graph with `max_depth=3`.

## 1. Setup

In [1]:
import sys
from pathlib import Path

# This tutorial is database-free: it loads the config and inspects the
# synthesized features and the generated SQL — none of which touch a
# database. To actually *execute* the features against PostgreSQL, run the
# example script instead (`just example <NN>`, or create_data.py +
# run_example.py with DATABASE_URL / PG* set). See the example README.
sys.path.insert(0, str(Path.cwd().parent.parent))

## 2. The Entity Graph

This example has a deeper entity graph than Examples 01 and 02. The relationships form a chain:

```
Stores ←── Orders ←── OrderItems ──→ Products ←── Suppliers
```

With `max_depth=3`, Featurizer traverses all the way from Stores down to Suppliers.

In [2]:
with open("config.yaml") as f:
    print(f.read())

# Retail supply chain: Deep nesting example

target: stores
max_depth: 3

intervals:
  - P30D  # Last 30 days
  - P90D  # Last 90 days

# Deep nesting multiplies features fast across 5 entities, so keep a minimal
# primitive set — the point here is the depth-3 traversal, not primitive breadth.
# (The full default set would blow past PostgreSQL's 1664 columns-per-row limit.)
aggregations:
  - count
  - sum
  - mean
transformations:
  - identity

entities:
  - alias: stores
    id: store_id
    table: stores
    temporal_ix: open_date
    variables:
      region:
        type: categorical
      size_sqft:
        type: numeric

  - alias: orders
    id: order_id
    table: orders
    temporal_ix: order_date
    variables:
      status:
        type: categorical

  - alias: order_items
    id: item_id
    table: order_items
    variables:
      quantity:
        type: numeric
      unit_price:
        type: numeric

  - alias: products
    id: product_id
    table: products
    variables:

## 3. Create Featurizer

Let's load the configuration and see how the entity graph is structured.

In [3]:
from featurizer import Featurizer

featurizer = Featurizer("config.yaml")

print(f"Target entity: {featurizer.target.alias}")
print(f"Max depth: {featurizer.max_depth}")
print(f"Intervals: {featurizer.intervals}")
print(f"Entities: {len(list(featurizer.entities))}")
print(f"Relationships: {len(featurizer.relationships)}")

2026-06-21 13:29:17.742 | DEBUG    | featurizer.planner:plan:230 - Starting feature build for target stores


2026-06-21 13:29:17.742 | DEBUG    | featurizer.planner:_build_features:256 - build_features(stores) depth=0


2026-06-21 13:29:17.742 | DEBUG    | featurizer.planner:_build_features:256 - build_features(orders) depth=1


2026-06-21 13:29:17.742 | DEBUG    | featurizer.planner:_build_features:256 - build_features(order_items) depth=2


2026-06-21 13:29:17.743 | INFO     | featurizer.planner:_build_features:276 - Maximum recursion depth reached at depth 3; materializing order_items without traversing further.


...

2026-06-21 13:29:17.744 | WARNING  | featurizer.planner:_build_aggregations:1027 - Entity Entity(order_items) lacks temporal index; skipping interval-based aggregation


2026-06-21 13:29:17.744 | WARNING  | featurizer.planner:_build_aggregations:1027 - Entity Entity(order_items) lacks temporal index; skipping interval-based aggregation


2026-06-21 13:29:17.744 | DEBUG    | featurizer.planner:_build_aggregations:1014 - Processing backward relationship Entity(stores).store_id -> Entity(orders).store_id


2026-06-21 13:29:17.745 | WARNING  | featurizer.planner:_apply_direct_roles:1155 - Direct variable 'stores.region' (type: categorical) will pass through as a raw string column and is likely to crash a downstream encoder. Set role: categorical (with a declared vocabulary or a PostgreSQL ENUM) to one-hot encode it, or role: identifier to exclude it.


Target entity: stores
Max depth: 3
Intervals: ['P30D', 'P90D']
Entities: 5
Relationships: 4


In [4]:
print("Entity Graph:")
for rel in featurizer.relationships:
    print(
        f"  {rel.parent.alias}.{rel.parent_key} ←── {rel.child.alias}.{rel.child_key}"
    )

Entity Graph:
  stores.store_id ←── orders.store_id
  orders.order_id ←── order_items.order_id
  order_items.product_id ←── products.product_id
  products.supplier_id ←── suppliers.supplier_id


## 4. Feature Propagation Across Depths

At each depth level, Featurizer aggregates child features and applies transformations. Let's examine how features are distributed across entities.

In [5]:
print("Features by entity:")
for entity_alias, features in featurizer.features.items():
    print(f"\n  {entity_alias}: {len(features)} features")
    sample = sorted(features, key=lambda f: f.name)[:5]
    for feat in sample:
        print(f"    - {feat.name}")

Features by entity:

  stores: 43 features
    - "COUNT(orders.order_date)"
    - "COUNT(orders.order_date|interval=P30D)"
    - "COUNT(orders.order_date|interval=P90D)"
    - "COUNT(orders.order_id)"
    - "COUNT(orders.order_id|interval=P30D)"

  orders: 47 features
    - "COUNT(order_items.item_id)"
    - "COUNT(orders.order_date)"
    - "COUNT(orders.order_date|interval=P30D)"
    - "COUNT(orders.order_date|interval=P90D)"
    - "COUNT(orders.order_id)"

  order_items: 8 features
    - "COUNT(order_items.item_id)"
    - "MEAN(order_items.quantity)"
    - "MEAN(order_items.unit_price)"
    - "SUM(order_items.quantity)"
    - "SUM(order_items.unit_price)"

  products: 3 features
    - base_cost
    - category
    - product_id

  suppliers: 3 features
    - country
    - rating
    - supplier_id


## 5. The CTE Chain

With deep nesting, the generated SQL has CTEs for each entity at each depth level. The naming pattern is:
- `<entity>_synth` — joins aggregated and direct features
- `<entity>_transform` — applies transformations
- `<child>_aggs_for_<parent>` — aggregation CTE

Let's see the full CTE structure.

In [6]:
print(f"Number of CTEs: {len(featurizer.ctes)}")
print("\nCTE names:")
for cte in featurizer.ctes:
    lines = cte.strip().split("\n")
    for line in lines:
        if " as (" in line:
            name = line.split(" as (")[0].strip().lstrip("-").strip()
            print(f"  - {name}")
            break

Number of CTEs: 8

CTE names:
  - order_items_synth
  - order_items_transform
  - order_items_aggs_for_orders
  - orders_synth
  - orders_transform
  - orders_aggs_for_stores
  - stores_synth
  - stores_transform


## 6. Generated SQL

The full SQL query shows the depth-3 traversal with lateral joins and time-windowed aggregations.

In [7]:
sql = featurizer.query
print("Generated SQL Query:")
print("=" * 80)
print(sql)
print("=" * 80)
print(f"\nSQL length: {len(sql):,} characters")

2026-06-21 13:29:17.757 | DEBUG    | featurizer.sql:render:40 - Rendered SQL for target 'stores': 8 CTEs, 14919 chars


Generated SQL Query:

        select aod.as_of_date, t.*
        from as_of_dates as aod
        cross join lateral (

        with

        
        -- sythetize aggregations and direct features for order_items
        order_items_synth as (
        select
        order_items.item_id, order_items.order_id, quantity, unit_price
        from order_items
        
        
        )
        ,
        -- transform order_items
        order_items_transform as (
        select
        item_id, order_id, quantity as quantity, unit_price as unit_price
        from order_items_synth _ego
        )
        ,
        -- Aggregate for orders
        order_items_aggs_for_orders as (
        select
        order_items_transform.order_id,
        count( item_id ) as "COUNT(order_items.item_id)" ,avg( quantity ) as "MEAN(order_items.quantity)" ,avg( unit_price ) as "MEAN(order_items.unit_price)" ,sum( quantity ) as "SUM(order_items.quantity)" ,sum( unit_price ) as "SUM(order_items.unit_price)" 
      

## 7. Feature Count Growth

Deep nesting causes a combinatorial explosion of features. Each depth level multiplies the feature count by the number of aggregations and transformations.

In [8]:
target_features = featurizer.features[featurizer.target.alias]
print(f"Total target features: {len(target_features)}")

# Analyze feature names
depth_indicators = {
    "Direct (stores)": [f for f in target_features if "orders" not in f.name.lower()],
    "Depth 1 (orders)": [
        f
        for f in target_features
        if "orders" in f.name.lower() and "order_items" not in f.name.lower()
    ],
    "Depth 2+ (items/products)": [
        f
        for f in target_features
        if "order_items" in f.name.lower() or "products" in f.name.lower()
    ],
}

for label, feats in depth_indicators.items():
    print(f"  {label}: ~{len(feats)} features")

Total target features: 43
  Direct (stores): ~4 features
  Depth 1 (orders): ~9 features
  Depth 2+ (items/products): ~30 features


## 8. Summary

In this tutorial, we learned:

1. **Deep entity graphs**: How to configure multi-level relationships (Stores → Orders → OrderItems → Products → Suppliers)
2. **Feature propagation**: How aggregated features at each depth level become inputs to the next level
3. **CTE chain**: The naming pattern for CTEs at each entity and depth
4. **Feature explosion**: How feature count grows combinatorially with depth, intervals, and primitives

### Performance Note
Deep nesting (depth > 3) can generate very large SQL queries. Consider:
- Reducing `max_depth` for initial exploration
- Using fewer `intervals`
- Selecting specific aggregations/transformations rather than defaults

In [9]:
print("Deep Nesting Summary")
print("=" * 40)
print(f"Target: {featurizer.target.alias}")
print(f"Depth: {featurizer.max_depth}")
print(f"Intervals: {', '.join(featurizer.intervals)}")
print(f"Entities: {len(list(featurizer.entities))}")
print(f"Relationships: {len(featurizer.relationships)}")
print(f"Total features: {len(target_features)}")
print(f"SQL length: {len(sql):,} characters")
print(f"CTEs generated: {len(featurizer.ctes)}")

Deep Nesting Summary
Target: stores
Depth: 3
Intervals: P30D, P90D
Entities: 5
Relationships: 4
Total features: 43
SQL length: 14,919 characters
CTEs generated: 8
